# Assignment 4: Linear Regression on Boston Housing

**Objective:** Train a linear regression model to predict the median value of owner-occupied homes in a Boston-area census tract.

The local `boston_housing.csv` file contains **506 observations**, **13 input features**, and the target **`medv`**. Despite its extension, it is whitespace-separated and has no header row. `medv` is measured in **thousands of historical US dollars**.

> **Dataset limitations:** Boston Housing is a historical teaching dataset with documented ethical and methodological concerns, including a racially derived feature (`b`). It was removed from scikit-learn's built-in datasets. This notebook reads the provided local file for the assignment; it should not be used for present-day housing valuation or decisions about people.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# 1. Load the local Boston Housing dataset (whitespace-separated, no header)
column_names = [
    "crim", "zn", "indus", "chas", "nox", "rm", "age",
    "dis", "rad", "tax", "ptratio", "b", "lstat", "medv"
]

# Support running from either the notebook folder or the repository root.
dataset_candidates = [
    Path("boston_housing.csv"),
    Path("AIML Lab") / "Assignment4" / "boston_housing.csv",
]
dataset_path = next((path for path in dataset_candidates if path.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError(
        "Cannot find boston_housing.csv. Run the notebook from its own folder "
        "or from the DTU-MTech-Lab repository root."
    )

df = pd.read_csv(dataset_path, sep=r"\s+", header=None)
if df.shape[1] != len(column_names):
    raise ValueError("Expected 14 columns: 13 Boston Housing features and medv.")
df.columns = column_names
if df.isna().any().any():
    raise ValueError("The Boston Housing dataset contains missing values.")

# 2. Separate the 13 input features (X) and target (y, in thousand USD)
X = df.drop("medv", axis=1)
y = df["medv"]

print(f"Dataset: {df.shape[0]} observations, {X.shape[1]} input features")
df.head()

In [ ]:
# 3. Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# 4. Create the Linear Regression model
model = LinearRegression()

In [ ]:
# 5. Train the model using only the training data
model.fit(X_train, y_train)

In [ ]:
# 6. Predict median home values for the held-out test data
y_pred = model.predict(X_test)

In [ ]:
# 7. Evaluate predictions on the held-out test set
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error ((thousand USD)^2): {mse:.4f}")
print(f"Root Mean Squared Error (thousand USD): {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

In [ ]:
# 8. Predict the median home value for a hypothetical Boston-area tract
# Use a DataFrame with the same 13 feature names and order as the training data.
new_house = pd.DataFrame([{
    "crim": 0.03,        # Per-capita crime rate
    "zn": 18.0,          # Residential land zoned for lots over 25,000 sq. ft.
    "indus": 2.31,       # Proportion of non-retail business acres
    "chas": 0,           # Charles River indicator (1 if tract borders river)
    "nox": 0.538,        # Nitric oxide concentration (parts per 10 million)
    "rm": 6.5,           # Average number of rooms per dwelling
    "age": 65.2,         # Percentage of owner-occupied units built before 1940
    "dis": 4.09,         # Weighted distance to five Boston employment centres
    "rad": 1,            # Index of accessibility to radial highways
    "tax": 296.0,        # Property-tax rate per $10,000
    "ptratio": 15.3,     # Pupil-teacher ratio by town
    "b": 396.9,          # Historical, racially derived variable (see note above)
    "lstat": 4.98,       # Percentage labelled "lower status" in the source dataset
}], columns=X.columns)

prediction = model.predict(new_house)[0]
print(f"Predicted median home value: {prediction:.2f} thousand USD")
print(f"Equivalent historical USD: ${prediction * 1000:,.2f}")

In [ ]:
# 9. Compare actual and predicted median home values (in thousand USD)
comparison = pd.DataFrame({
    "Actual MEDV": y_test,
    "Predicted MEDV": y_pred,
    "Residual (actual - predicted)": y_test - y_pred,
})
comparison.head(10)